## Estimating Impact during our closing trading window

In [1]:
from data_collection.data_adapters import DatabentoGenericAdapter
from dotenv import load_dotenv
from pathlib import Path
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
load_dotenv()
pl.enable_string_cache()

CACHE_DIR = Path.cwd().resolve().parents[0] / "data" / "dbn_cache" / "generic"
adapter = DatabentoGenericAdapter(os.getenv("DATABENTO-API-KEY"), CACHE_DIR)

print(f"CACHE_DIR: {CACHE_DIR}")
# Clear the cache if needed
# adapter.clear_cache()

with open("../universe.txt", "r") as f:
    UNIVERSE = [line.strip() for line in f]
with open("../top_50_universe.txt", "r") as f:
    ACTIVE_SET = [s for s in [line.strip() for line in f] if s not in ['CRCL', 'OMDA']]
START = "2025-06-11"
END = "2026-06-11"
DATASET = "XNAS.ITCH"

# trades with the bbo
df_trades = (
    adapter.scan_data(
        tickers=UNIVERSE, start_date=START, end_date=END,
        dataset=DATASET, schema="tbbo",
    )
    # --- Filters and sorts for symbol selection, time alignment, speed, and memory
    .select(["ts_event", "symbol", "price", "side", "size", "ask_px_00", "bid_px_00"])
    .filter(pl.col('ts_event').dt.time().is_between(pl.time(13,30), pl.time(21,0)),
            pl.col("symbol").is_in(ACTIVE_SET))
    .with_columns(pl.col('ts_event').dt.convert_time_zone("America/New_York"))
    .filter(pl.col('ts_event').dt.time().is_between(pl.time(15,50), pl.time(16,0)))
    .with_columns(
        ((pl.col("ask_px_00") + pl.col("bid_px_00")) / 2).alias("mid"),
        pl.col("symbol").cast(pl.Categorical),
        pl.col("ts_event").dt.date().alias("date"),
    )
    .filter(pl.col("date").is_between(pl.date(2025,6,11), pl.date(2026,1,1)))
    .drop(["ask_px_00", "bid_px_00"])
    .sort("symbol", "ts_event")
    # --- Trade Signing ---
    # quote rule (only needed when side == "N") and tick rule (for mid trades)
    .with_columns(
        pl.when(pl.col("price") > pl.col("mid")).then(1)
          .when(pl.col("price") < pl.col("mid")).then(-1)
          .otherwise(None).alias("quote_sign"),
        pl.when(pl.col("price").diff().over("symbol") > 0).then(1)
          .when(pl.col("price").diff().over("symbol") < 0).then(-1)
          .otherwise(None).alias("tick_sign"),
    )
    .with_columns(pl.col("tick_sign").forward_fill().over("symbol"))
    # First trust the given sign, then use Lee-Ready
    .with_columns(
        pl.when(pl.col("side") == "B").then(1)
          .when(pl.col("side") == "A").then(-1)
          .otherwise(pl.coalesce("quote_sign", "tick_sign"))
          .alias("trade_sign")
    )
)

CACHE_DIR: /Users/michaelcusack-nelkin/GitHub/Summer_Research_2026/data/dbn_cache/generic
[GenericAdapter] scan_parquet: 6436dd54a7ced32262a3242e08d42d20.parquet


In [2]:
data = (
    df_trades
    .group_by_dynamic(
        index_column="ts_event",
        every="1s",
        closed="right",
        group_by="symbol"
    )
    .agg(
        pl.col("mid").last(),
        (pl.col("trade_sign") * pl.col("size") * pl.col("price")).sum().alias("signed_dollar_flow")
    )
    .with_columns(pl.col("mid").pct_change().over("symbol").alias("ret"))
    .select(["ts_event", "symbol", "signed_dollar_flow", "ret"])
    .drop_nulls()
    .collect(engine="streaming")
)

In [3]:
data

ts_event,symbol,signed_dollar_flow,ret
"datetime[ns, America/New_York]",cat,f64,f64
2025-06-11 15:50:01 EDT,"""GOOGL""",150742.51,0.000225
2025-06-11 15:50:02 EDT,"""GOOGL""",-81490.17,0.000056
2025-06-11 15:50:03 EDT,"""GOOGL""",74572.155,0.0
2025-06-11 15:50:04 EDT,"""GOOGL""",-16688.82,-0.000169
2025-06-11 15:50:05 EDT,"""GOOGL""",155175.33,0.000169
…,…,…,…
2025-12-31 15:59:55 EST,"""NFLX""",883707.35,0.000267
2025-12-31 15:59:56 EST,"""NFLX""",-241243.425,0.0
2025-12-31 15:59:57 EST,"""NFLX""",-561334.09,0.0


In [4]:
# --- Kyle regression: ret ~ signed $ flow, per symbol.  slope = gamma_i (return per $) ---
# gamma_i = Cov(ret, flow) / Var(flow) is the OLS slope (with intercept). Kyle's lambda per name.
MIN_OBS = 500   # drop names with too few active 1s buckets to fit a stable slope

kyle = (
    data.group_by("symbol")
    .agg(
        (pl.cov("ret", "signed_dollar_flow") / pl.col("signed_dollar_flow").var()).alias("gamma"),
        pl.corr("ret", "signed_dollar_flow").alias("corr"),
        pl.len().alias("n"),
    )
    .with_columns((pl.col("corr") ** 2).alias("r2"))
    .filter(pl.col("n") >= MIN_OBS)
    .sort("gamma", descending=True)
)

# Assemble a gamma vector aligned to ACTIVE_SET (NaN = not fit / dropped by MIN_OBS)
gamma_map = dict(zip(kyle["symbol"].cast(pl.Utf8).to_list(), kyle["gamma"].to_list()))
GAMMA_VEC = np.abs(np.array([gamma_map.get(s, np.nan) for s in ACTIVE_SET]))

n_neg = int((kyle["gamma"] < 0).sum())
print(f"fit {kyle.height}/{len(ACTIVE_SET)} names | wrong-signed (gamma<0): {n_neg} "
      f"| median gamma: {np.nanmedian(GAMMA_VEC):.3e} return/$")

kyle

fit 47/49 names | wrong-signed (gamma<0): 3 | median gamma: 1.625e-10 return/$


symbol,gamma,corr,n,r2
cat,f64,f64,u64,f64
"""MSTR""",6.7290e-10,0.139294,59917,0.019403
"""BABA""",6.6069e-10,0.029343,41180,0.000861
"""COIN""",5.8410e-10,0.076354,54704,0.00583
"""HOOD""",5.5369e-10,0.067933,63292,0.004615
"""SMCI""",4.8515e-10,0.060229,52435,0.003628
…,…,…,…,…
"""SPY""",1.7175e-11,0.0722,79827,0.005213
"""VOO""",9.3120e-12,0.007653,48558,0.000059
"""TQQQ""",-4.7445e-12,-0.000544,58295,2.9635e-7


In [5]:
med = np.nanmedian(GAMMA_VEC)
GAMMA_VEC = np.where(np.isnan(GAMMA_VEC), med, GAMMA_VEC)
np.save("impact_gamma_vec.npy", GAMMA_VEC)
GAMMA_VEC

array([6.12030438e-11, 1.62410272e-10, 1.71751918e-11, 2.83055048e-11,
       5.93373465e-11, 3.48253309e-11, 4.29618877e-11, 9.03746576e-11,
       1.26924127e-10, 8.94587779e-11, 3.06182182e-10, 6.72896566e-10,
       1.49610859e-10, 6.88834677e-11, 4.74450253e-12, 2.49922856e-11,
       2.57067204e-10, 1.09694245e-10, 5.87810181e-11, 5.84100509e-10,
       3.17776352e-10, 1.69203173e-10, 1.62469051e-10, 4.85145317e-10,
       3.50215083e-10, 7.25611644e-11, 2.01180248e-10, 1.99036787e-10,
       3.35402685e-10, 5.53689528e-10, 1.31140769e-10, 6.24968568e-10,
       8.96743301e-11, 3.94924189e-10, 2.42658426e-10, 1.95461033e-10,
       2.54122821e-10, 7.70760300e-11, 1.21610634e-10, 3.40126276e-10,
       1.28316753e-10, 6.60687378e-10, 4.48879300e-10, 1.65714607e-10,
       3.65202352e-10, 1.62469051e-10, 3.87361686e-10, 1.62469051e-10,
       9.31198337e-12])

In [6]:
fit = set(kyle["symbol"].cast(pl.Utf8).to_list())
missing = [s for s in ACTIVE_SET if s not in fit]
print(f"{len(missing)} of {len(ACTIVE_SET)} not fit:")
print(missing)

2 of 49 not fit:
['IBIT', 'CRWV']


In [7]:
counts = (data.group_by("symbol").len()
          .filter(pl.col("symbol").cast(pl.Utf8).is_in(["IBIT", "CRWV"])))
print(counts)
print("MIN_OBS =", 500)

shape: (0, 2)
┌────────┬─────┐
│ symbol ┆ len │
│ ---    ┆ --- │
│ cat    ┆ u64 │
╞════════╪═════╡
└────────┴─────┘
MIN_OBS = 500


In [8]:
for s in ["IBIT", "CRWV"]:
    print(f"{s}: in UNIVERSE={s in UNIVERSE}, in ACTIVE_SET={s in ACTIVE_SET}")

# do they exist at all in the raw tbbo parquet (before the 15:50-16:00 / date filters)?
raw = (
    adapter.scan_data(tickers=UNIVERSE, start_date=START, end_date=END,
                      dataset=DATASET, schema="tbbo")
    .select(["symbol"])
    .filter(pl.col("symbol").is_in(["IBIT", "CRWV"]))
    .group_by("symbol").len()
    .collect(engine="streaming")
)
print(raw)

IBIT: in UNIVERSE=False, in ACTIVE_SET=True
CRWV: in UNIVERSE=False, in ACTIVE_SET=True
[GenericAdapter] scan_parquet: 6436dd54a7ced32262a3242e08d42d20.parquet
shape: (0, 2)
┌────────┬─────┐
│ symbol ┆ len │
│ ---    ┆ --- │
│ str    ┆ u64 │
╞════════╪═════╡
└────────┴─────┘
